In [ ]:
import pandas as pd
import numpy as np
import joblib
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix
from sklearn.preprocessing import MinMaxScaler, LabelEncoder
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from xgboost import XGBClassifier
import warnings

warnings.filterwarnings('ignore')

def calculate_metrics(y_true, y_pred, y_proba):
    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, average='macro', zero_division=0)
    rec = recall_score(y_true, y_pred, average='macro', zero_division=0)
    f1 = f1_score(y_true, y_pred, average='macro', zero_division=0)
    
    cm = confusion_matrix(y_true, y_pred)
    fp = cm.sum(axis=0) - np.diag(cm)
    fn = cm.sum(axis=1) - np.diag(cm)
    tp = np.diag(cm)
    tn = cm.sum() - (fp + fn + tp)
    
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        fpr_array = fp / (fp + tn)
        fpr = np.nanmean(fpr_array)
    
    if y_proba is not None:
        try:
            auc = roc_auc_score(y_true, y_proba, multi_class='ovr', average='macro')
        except ValueError:
            auc = np.nan
    else:
        auc = np.nan
        
    return round(acc, 4), round(prec, 4), round(rec, 4), round(f1, 4), round(auc, 4), round(fpr, 4)

def plot_cm(y_true, y_pred, classes, title, filename):
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(12, 10), dpi=300)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=classes, yticklabels=classes)
    plt.title(title, fontsize=14)
    plt.ylabel('Rzeczywista klasa', fontsize=12)
    plt.xlabel('Przewidziana klasa', fontsize=12)
    plt.tight_layout()
    plt.savefig(filename)
    plt.close()

def run_optimization_and_evaluation():
    print("Inicjalizacja przestrzeni i standaryzacja zbiorów...")
    train_clean = pd.read_csv('UNSW_NB15_training-set.csv').drop(columns=['id', 'label'], errors='ignore')
    test_clean = pd.read_csv('UNSW_NB15_testing-set.csv').drop(columns=['id', 'label'], errors='ignore')

    for df in [train_clean, test_clean]:
        df['sbytes_per_packet'] = df['sbytes'] / (df['spkts'] + 1e-5)
        df['dbytes_per_packet'] = df['dbytes'] / (df['dpkts'] + 1e-5)
        df['sbytes_per_second'] = df['sbytes'] / (df['dur'] + 1e-5)
        df['bytes_ratio_s_to_d'] = df['sbytes'] / (df['dbytes'] + 1e-5)
        
    optimal_features = [
        'proto_udp', 'service_irc', 'sttl', 'service_dns', 'bytes_ratio_s_to_d',
        'sbytes', 'ct_srv_dst', 'dbytes', 'ct_state_ttl', 'sbytes_per_packet',
        'ct_dst_src_ltm', 'ct_dst_sport_ltm', 'ct_srv_src', 'smean', 'dbytes_per_packet',
        'synack', 'ackdat', 'dmean', 'state_INT', 'dttl',
        'dload', 'dur', 'sbytes_per_second', 'ct_src_dport_ltm', 'rate'
    ]

    skewed_features = [
        'sbytes', 'dbytes', 'dur', 'rate', 'dload', 
        'sbytes_per_packet', 'dbytes_per_packet', 'sbytes_per_second', 'bytes_ratio_s_to_d'
    ]
    
    for col in skewed_features:
        if col in train_clean.columns:
            train_clean[col] = np.log1p(train_clean[col])
            test_clean[col] = np.log1p(test_clean[col])

    y_train = train_clean['attack_cat']
    y_test = test_clean['attack_cat']

    le = LabelEncoder()
    le.fit(y_train.str.strip().str.lower().replace('backdoors', 'backdoor'))
    y_test_enc = np.array([le.transform([x])[0] if x in le.classes_ else 0 for x in y_test.str.strip().str.lower().replace('backdoors', 'backdoor')])
    class_names = le.classes_

    X_train = train_clean.drop(columns=['attack_cat'])
    X_test = test_clean.drop(columns=['attack_cat'])

    cat_cols = X_train.select_dtypes(include=['object', 'category']).columns.tolist()
    X_train_dum = pd.get_dummies(X_train, columns=cat_cols, drop_first=False)
    X_test_dum = pd.get_dummies(X_test, columns=cat_cols, drop_first=False)
    
    for col in optimal_features:
        if col not in X_train_dum.columns:
            X_train_dum[col] = 0
        if col not in X_test_dum.columns:
            X_test_dum[col] = 0
            
    X_train_opt = X_train_dum[optimal_features]
    X_test_opt = X_test_dum[optimal_features]

    scaler = MinMaxScaler()
    scaler.fit(X_train_opt)
    X_real_test = pd.DataFrame(scaler.transform(X_test_opt), columns=optimal_features)

    print("Wczytywanie zbalansowanego zbioru treningowego (GAN)...")
    df_gan = pd.read_csv('UNSW_NB15_GAN_Balanced_main.csv')
    X_gan = df_gan[optimal_features]
    y_gan = df_gan['target_class']

    X_gan_train, X_gan_test, y_gan_train, y_gan_test = train_test_split(X_gan, y_gan, test_size=0.2, random_state=42, stratify=y_gan)

    models = {
        'Naive_Bayes': (GaussianNB(), {'var_smoothing': np.logspace(-11, -8, num=10)}),
        'Logistic_Regression': (LogisticRegression(max_iter=1000, n_jobs=-1, random_state=42), {'C': [0.01, 0.1, 0.5, 1, 5, 10, 20, 50, 100, 200]}),
        'Linear_SVM': (LinearSVC(random_state=42, max_iter=1000, dual=False), {'C': [0.01, 0.1, 0.5, 1, 5, 10, 20, 50, 100, 200]}),
        'KNN': (KNeighborsClassifier(n_jobs=-1), {'n_neighbors': [3, 5, 7, 9, 11, 13, 15, 17, 19, 21]}),
        'Decision_Tree': (DecisionTreeClassifier(random_state=42), {'max_depth': [10, 20, 30, None, 40, 50], 'min_samples_split': [2, 5, 10, 15, 20]}),
        'Extra_Trees': (ExtraTreesClassifier(random_state=42, n_jobs=-1), {'n_estimators': [100, 200, 300], 'max_depth': [10, 20, 30, None], 'min_samples_split': [2, 5, 10]}),
        'Random_Forest': (RandomForestClassifier(random_state=42, n_jobs=-1), {'n_estimators': [100, 200, 300], 'max_depth': [10, 20, 30, None], 'min_samples_split': [2, 5, 10]}),
        'XGBoost': (XGBClassifier(eval_metric='mlogloss', random_state=42, n_jobs=-1), {'n_estimators': [100, 200, 300], 'learning_rate': [0.01, 0.05, 0.1, 0.2], 'max_depth': [5, 7, 9], 'subsample': [0.8, 1.0],'reg_lambda': [0.1, 1.0, 5.0, 10.0]})
    }

    results = []

    print("Rozpoczęcie optymalizacji hiperparametrów, generacji macierzy i ewaluacji...\n")
    for name, (base_model, param_dist) in models.items():
        print(f" -> Przetwarzanie modelu: {name}")
        search = RandomizedSearchCV(
            base_model, param_distributions=param_dist, 
            n_iter=10, scoring='f1_macro', cv=3, random_state=42, n_jobs=-1
        )
        search.fit(X_gan_train, y_gan_train)
        
        best_model = search.best_estimator_
        joblib.dump(best_model, f'Model_{name}_Final.pkl')
        
        y_pred_gan = best_model.predict(X_gan_test)
        y_proba_gan = best_model.predict_proba(X_gan_test) if hasattr(best_model, "predict_proba") else None
        acc_g, prec_g, rec_g, f1_g, auc_g, fpr_g = calculate_metrics(y_gan_test, y_pred_gan, y_proba_gan)
        
        plot_cm(y_gan_test, y_pred_gan, class_names, f"Macierz pomyłek - {name} (GAN 20%)", f"CM_{name}_GAN.png")
        
        results.append({
            'Model': name,
            'Zbiór': 'GAN_Test_20%',
            'Accuracy': acc_g, 'Precision': prec_g, 'Recall': rec_g, 'F1_Macro': f1_g, 'ROC_AUC': auc_g, 'FPR_Macro': fpr_g
        })
        
        y_pred_real = best_model.predict(X_real_test)
        y_proba_real = best_model.predict_proba(X_real_test) if hasattr(best_model, "predict_proba") else None
        acc_r, prec_r, rec_r, f1_r, auc_r, fpr_r = calculate_metrics(y_test_enc, y_pred_real, y_proba_real)
        
        plot_cm(y_test_enc, y_pred_real, class_names, f"Macierz pomyłek - {name} (Zbiór Natywny)", f"CM_{name}_Native.png")
        
        results.append({
            'Model': name,
            'Zbiór': 'Native_Testing_Set',
            'Accuracy': acc_r, 'Precision': prec_r, 'Recall': rec_r, 'F1_Macro': f1_r, 'ROC_AUC': auc_r, 'FPR_Macro': fpr_r
        })

    results_df = pd.DataFrame(results)
    print("\n--- ZESTAWIENIE WYNIKÓW ---")
    print(results_df.to_string(index=False))
    results_df.to_csv('Wyniki_Trening_Ewaluacja_Zestawienie.csv', index=False)
    print("\nZapisano wyniki oraz macierze pomyłek do plików.")

run_optimization_and_evaluation()